# Jigsaw · Historical lexical reference

This preserves the original TF-IDF/logistic model that scored **0.59191 public / 0.61956 private** in Kaggle Version 2. It is retained for reproducibility and CPU software checks.

Use [submission.ipynb](submission.ipynb) for the support-adapted GPU method. The two notebooks have different model purposes.

This reference works in SageMaker/Jupyter or Kaggle. With `GENERATE_SUBMISSION = True`, it fits the original training rows, generates the preview CSV, validates the output and provides a download. Completed model fits and prediction batches are checksummed and reusable. No upload happens inside the notebook.

The separate 0.7770 post-competition research result is not this notebook's Kaggle score. A local ten-row preview is not an independent performance evaluation.

In [ ]:
from __future__ import annotations
import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'

## Canonical schema, model, and resumable runtime
These cells are generated from the tested package modules. No downloads, external model calls, or Kaggle API calls are required.

In [ ]:
"""Competition schemas and deterministic synthetic data for software verification."""


import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

EXAMPLES = ["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"]
TEXT = ["body", "rule", "subreddit", *EXAMPLES]
FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()


def validate_frame(df: pd.DataFrame, *, train: bool) -> None:
    required = ["row_id", *TEXT] + (["rule_violation"] if train else [])
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if df.empty or df.row_id.isna().any() or df.row_id.duplicated().any():
        raise ValueError("Rows and unique, non-null row_id values are required")
    for column in TEXT:
        if not df[column].map(lambda v: isinstance(v, str) and bool(v.strip())).all():
            raise ValueError(f"Empty or non-text values in {column}")
    if train and (df.rule_violation.isna().any() or not df.rule_violation.isin([0, 1]).all()):
        raise ValueError("Targets must be binary 0/1 with no missing values")
    if not train and "rule_violation" in df:
        raise ValueError("Test data must not contain target labels")


def validate_submission(submission: pd.DataFrame, sample: pd.DataFrame) -> None:
    if list(submission.columns) != ["row_id", "rule_violation"]:
        raise ValueError("Submission must have exactly row_id,rule_violation")
    if submission.empty or submission.row_id.isna().any() or submission.row_id.duplicated().any():
        raise ValueError("Submission IDs must be unique and non-null")
    if not submission.row_id.equals(sample.row_id):
        raise ValueError("Submission row IDs or order differ from sample")
    p = submission.rule_violation.to_numpy(dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Predictions must be finite probabilities in [0,1]")


def load_data(directory: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    for name in FILES:
        if not (directory / name).is_file():
            raise FileNotFoundError(f"Missing {directory / name}; run jigsaw download first")
    train, test, sample = (pd.read_csv(directory / name) for name in FILES)
    validate_frame(train, train=True)
    validate_frame(test, train=False)
    if list(sample.columns) != ["row_id", "rule_violation"]:
        raise ValueError("Unexpected sample submission columns")
    if not test.row_id.equals(sample.row_id):
        raise ValueError("Test and sample submission IDs/order differ")
    validate_submission(sample, sample)
    if set(train.row_id) & set(test.row_id):
        raise ValueError("Train and test row IDs overlap")
    return train, test, sample


def audit(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    train_bodies = set(train.body.map(normalize))
    test_bodies = set(test.body.map(normalize))
    return {
        "train_rows": len(train),
        "preview_test_rows": len(test),
        "train_rules": sorted(train.rule.unique().tolist()),
        "preview_test_rules": sorted(test.rule.unique().tolist()),
        "duplicate_training_bodies": int(train.body.map(normalize).duplicated().sum()),
        "train_test_body_overlap": len(train_bodies & test_bodies),
        "label_prevalence": float(train.rule_violation.mean()),
        "by_rule": train.groupby("rule")
        .rule_violation.agg(["size", "mean"])
        .reset_index()
        .to_dict("records"),
        "test_note": "Downloaded test is a preview; hidden evaluation can replace it.",
    }


def synthetic(directory: Path) -> None:
    """Tiny authored examples; never use these metrics as competition performance."""
    directory.mkdir(parents=True, exist_ok=True)
    if any((directory / f).exists() for f in FILES):
        raise FileExistsError("Synthetic generation refuses to overwrite existing data")
    rows = []
    for r, rule in enumerate(["No advertisements", "No personal insults"]):
        for i in range(48):
            label = i % 2
            phrase = (
                ["buy discount offer", "you stupid fool"][r]
                if label
                else "thoughtful topic discussion"
            )
            rows.append(
                {
                    "row_id": r * 100 + i,
                    "body": f"{phrase} uniqueitem{r}x{i}",
                    "rule": rule,
                    "subreddit": f"community{i % 4}",
                    "positive_example_1": "buy discount coupon" if r == 0 else "you foolish idiot",
                    "positive_example_2": "sale offer now" if r == 0 else "stupid personal insult",
                    "negative_example_1": "thank you for this thoughtful discussion",
                    "negative_example_2": "interesting topic worth discussing",
                    "rule_violation": label,
                }
            )
    train = pd.DataFrame(rows)
    test = train.iloc[:8].drop(columns="rule_violation").copy()
    test["row_id"] = np.arange(1000, 1008)
    test["body"] = test.body + " unseen"
    sample = pd.DataFrame({"row_id": test.row_id, "rule_violation": 0.5})
    for frame, name in zip([train, test, sample], FILES, strict=True):
        frame.to_csv(directory / name, index=False)
    (directory / "SYNTHETIC.txt").write_text("SOFTWARE TEST DATA. NOT COMPETITION RESULTS.\n")


In [ ]:
"""CPU reference models with explicit comment/example interactions."""


import warnings

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



class LexicalClassifier:
    """A transparent reference point; this model is not claimed to understand new policies."""

    def __init__(self, context: bool = True, seed: int = 2025):
        self.context = context
        self.seed = seed
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), sublinear_tf=True, max_features=40000, dtype=np.float64
        )
        self.classifier = LogisticRegression(
            C=2.0, solver="liblinear", max_iter=2000, random_state=seed
        )

    def _features(self, frame: pd.DataFrame):
        body = self.vectorizer.transform(frame.body)
        if not self.context:
            return body
        sims = [
            np.asarray(body.multiply(self.vectorizer.transform(frame[c])).sum(axis=1)).ravel()
            for c in ["rule", *EXAMPLES]
        ]
        pos = np.maximum(sims[1], sims[2])
        neg = np.maximum(sims[3], sims[4])
        extra = np.column_stack([*sims, pos, neg, pos - neg])
        return hstack([body, csr_matrix(extra)], format="csr")

    def fit(self, frame: pd.DataFrame) -> LexicalClassifier:
        columns = ["body", "rule", *EXAMPLES] if self.context else ["body"]
        corpus = [text for column in columns for text in frame[column]]
        self.vectorizer.fit(corpus)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            self.classifier.fit(self._features(frame), frame.rule_violation)
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        return self.classifier.predict_proba(self._features(frame))[:, 1]

    def coefficients(self) -> pd.DataFrame:
        names = self.vectorizer.get_feature_names_out().tolist()
        if self.context:
            names += [
                "similarity_rule",
                *[f"similarity_{c}" for c in EXAMPLES],
                "max_positive_similarity",
                "max_negative_similarity",
                "similarity_margin",
            ]
        return pd.DataFrame({"feature": names, "coefficient": self.classifier.coef_[0]})


In [ ]:
"""Atomic stage commits, checksummed reuse, and UTC progress events."""


import hashlib
import json
import os
import platform
import shutil
import tempfile
import threading
import time
from collections.abc import Callable
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

from filelock import FileLock


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def atomic_bytes(path: Path, value: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".partial")
    with tmp.open("wb") as stream:
        stream.write(value)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(tmp, path)


def atomic_json(path: Path, value: Any) -> None:
    atomic_bytes(path, (json.dumps(value, indent=2, allow_nan=False) + "\n").encode())


def environment() -> dict:
    packages = ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "plotly"]
    return {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}


def fingerprint(data_dir: Path, config: dict) -> tuple[str, dict]:
    source = Path(__file__).parent
    record = {
        "data": {p.name: digest(p) for p in sorted(data_dir.glob("*.csv"))},
        "code": {p.name: digest(p) for p in sorted(source.glob("*.py"))},
        "config": config,
        "environment": environment(),
    }
    key = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()[:20]
    return key, record


class Progress:
    """Every long operation emits start, heartbeat, finish/failure, and elapsed seconds."""

    _context = threading.local()

    def __init__(
        self,
        path: Path,
        stage: str,
        heartbeat_seconds: float = 15,
        *,
        total_started: float | None = None,
    ):
        self.path = path
        self.stage = stage
        self.interval = heartbeat_seconds
        self.started = time.monotonic()
        self.total_started = self.started if total_started is None else total_started
        self.explicit_total_started = total_started
        self.stop = threading.Event()
        self.guard = threading.Lock()
        self.thread: threading.Thread | None = None

    def emit(self, event: str, **fields: Any) -> None:
        record = {
            "timestamp": datetime.now(UTC).isoformat(timespec="seconds"),
            "stage": self.stage,
            "event": event,
            "elapsed_seconds": round(time.monotonic() - self.started, 3),
            "stage_elapsed_seconds": round(time.monotonic() - self.started, 3),
            "total_elapsed_seconds": round(time.monotonic() - self.total_started, 3),
            **fields,
        }
        line = json.dumps(record, allow_nan=False)
        with self.guard:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with self.path.open("a", encoding="utf-8") as stream:
                stream.write(line + "\n")
                stream.flush()
            print(line, flush=True)

    def _heartbeat(self) -> None:
        while not self.stop.wait(self.interval):
            self.emit("heartbeat")

    def __enter__(self) -> Progress:
        self.previous_total_started = getattr(self._context, "started", None)
        inherited = self.previous_total_started
        if self.explicit_total_started is None and inherited is not None:
            self.total_started = inherited
        self._context.started = self.total_started
        self.emit("started")
        self.thread = threading.Thread(target=self._heartbeat, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, kind, error, traceback) -> None:
        self.stop.set()
        if self.thread:
            self.thread.join(timeout=2)
        try:
            self.emit(
                "failed" if error else "completed", error_type=kind.__name__ if kind else None
            )
        finally:
            self._context.started = self.previous_total_started


def stage(directory: Path, name: str, action: Callable[[Path], None]) -> Path:
    """A failed stage restarts; completed, intact stages are reused without recomputing."""
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / name
    with FileLock(str(directory / f"{name}.lock"), timeout=1):
        with Progress(directory / "events.jsonl", name) as log:
            marker = target / "complete.json"
            if marker.exists():
                record = json.loads(marker.read_text())
                intact = bool(record["files"]) and all(
                    (target / p).is_file() and digest(target / p) == sha
                    for p, sha in record["files"].items()
                )
                if intact:
                    log.emit("reused", files=len(record["files"]))
                    return target
                log.emit("recomputing", reason="output checksum mismatch")
            # Only committed directories count as checkpoints. Abandoned work is isolated.
            with tempfile.TemporaryDirectory(prefix=f".{name}-", dir=directory) as temporary:
                work = Path(temporary) / "artifacts"
                work.mkdir()
                action(work)
                files = {
                    str(p.relative_to(work)): digest(p)
                    for p in sorted(work.rglob("*"))
                    if p.is_file() and not p.name.endswith(".partial")
                }
                if not files:
                    raise ValueError(f"Stage {name} produced no artifacts")
                atomic_json(
                    work / "complete.json",
                    {"files": files, "finished_at": datetime.now(UTC).isoformat()},
                )
                if target.exists():
                    shutil.rmtree(target)
                os.replace(work, target)
            return target


In [ ]:
"""User-run offline inference with durable model/batch checkpoints and verified downloads."""


import base64
import hashlib
import html
import json
import platform
from importlib.metadata import version
from pathlib import Path

import joblib
import pandas as pd
from filelock import FileLock



def generate_submission(
    input_root: Path,
    output_root: Path,
    cache_root: Path,
    *,
    source_sha256: str,
    batch_size: int = 5000,
    seed: int = 2025,
) -> Path:
    """Fit once, resume completed prediction batches, validate, then publish locally.

    Only call this for the user's explicit generation action or synthetic software tests.
    It never calls the Kaggle API. Cache files are trusted locally created artifacts.
    """
    if (
        not isinstance(batch_size, int)
        or isinstance(batch_size, bool)
        or batch_size < 1
        or not isinstance(source_sha256, str)
        or len(source_sha256) != 64
        or any(c not in "0123456789abcdef" for c in source_sha256)
    ):
        raise ValueError("Positive batch_size and a full source SHA-256 are required")
    input_root, output_root, cache_root = map(Path, (input_root, output_root, cache_root))
    inputs = {name: digest(input_root / name) for name in FILES}
    train, test, sample = load_data(input_root)
    contract = {
        "schema": 1,
        "source_sha256": source_sha256,
        "inputs": inputs,
        "synthetic": (input_root / "SYNTHETIC.txt").exists(),
        "model": "rule_examples",
        "seed": seed,
        "batch_size": batch_size,
        "python": platform.python_version(),
        "packages": {p: version(p) for p in ("numpy", "pandas", "scipy", "scikit-learn", "joblib")},
    }
    key = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()
    directory = cache_root / key
    output_root.mkdir(parents=True, exist_ok=True)
    cache_root.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output_root / "submission.lock"), timeout=1):
        with FileLock(str(cache_root / f"{key}.lock"), timeout=1):
            with Progress(directory / "events.jsonl", "offline_submission") as log:

                def unchanged() -> None:
                    if inputs != {name: digest(input_root / name) for name in FILES}:
                        raise ValueError("Input files changed during generation; export refused")

                unchanged()
                atomic_json(directory / "contract.json", contract)
                log.emit("data_validated", train_rows=len(train), test_rows=len(test))

                def fit(destination: Path) -> None:
                    fitted = LexicalClassifier(context=True, seed=seed).fit(train)
                    joblib.dump(fitted, destination / "model.joblib")

                fitted_path = stage(directory, "model", fit)
                fitted = None
                chunks = []
                for offset in range(0, len(test), batch_size):
                    end = min(offset + batch_size, len(test))

                    def predict(destination: Path, start=offset, stop=end) -> None:
                        nonlocal fitted
                        if fitted is None:
                            fitted = joblib.load(fitted_path / "model.joblib")
                        frame = pd.DataFrame(
                            {
                                "row_id": test.iloc[start:stop].row_id.to_numpy(),
                                "rule_violation": fitted.predict(test.iloc[start:stop]),
                            }
                        )
                        validate_submission(frame, sample.iloc[start:stop].reset_index(drop=True))
                        atomic_bytes(
                            destination / "predictions.csv", frame.to_csv(index=False).encode()
                        )

                    part = stage(directory, f"batch_{offset:09d}", predict)
                    frame = pd.read_csv(part / "predictions.csv")
                    validate_submission(frame, sample.iloc[offset:end].reset_index(drop=True))
                    chunks.append(frame)
                    log.emit("prediction_batch", completed_rows=end, total_rows=len(test))
                submission = pd.concat(chunks, ignore_index=True)
                validate_submission(submission, sample)
                unchanged()
                payload = submission.to_csv(index=False).encode()
                manifest = {
                    **contract,
                    "contract_sha256": key,
                    "rows": len(submission),
                    "submission_sha256": hashlib.sha256(payload).hexdigest(),
                    "status": "Validated local inference; no Kaggle score or upload.",
                }
                atomic_bytes(output_root / "submission.csv", payload)
                atomic_json(output_root / "submission_manifest.json", manifest)
                log.emit("SUBMISSION_VALIDATED", rows=len(submission), contract=key)
    return output_root / "submission.csv"


def download_link(path: Path, *, max_bytes: int = 10_000_000) -> str:
    """Return a click-to-download HTML link only for a checksum-verified generated CSV.

    A small data URI avoids fragile Jupyter/SageMaker proxy-relative URLs. Large files
    use the notebook file browser rather than embedding an unbounded payload.
    """
    path = Path(path)
    manifest = json.loads((path.parent / "submission_manifest.json").read_text())
    if path.name != "submission.csv" or digest(path) != manifest.get("submission_sha256"):
        raise ValueError("Submission checksum mismatch; regenerate before downloading")
    if path.stat().st_size > max_bytes:
        return (
            "<p>Validated file is large. Download submission.csv from the output file browser.</p>"
        )
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != manifest["submission_sha256"]:
        raise ValueError("Submission changed while creating the download link")
    data = base64.b64encode(payload).decode("ascii")
    name = html.escape(path.name, quote=True)
    return f'<a download="{name}" href="data:text/csv;base64,{data}">Download {name}</a>'

## Generate locally
`GENERATE_SUBMISSION` controls this action. Running with `True` creates or resumes your own output; `False` performs no inference. Paths are detected from the project or Kaggle environment. The source/data/environment fingerprint prevents stale checkpoint reuse.

In [ ]:
"""Resolve the official competition mount across Kaggle runtime layouts."""

import os
from pathlib import Path


def submission_input_root(local_root: Path, kaggle_root: Path = Path("/kaggle/input")) -> Path:
    override = os.environ.get("JIGSAW_KAGGLE_INPUT")
    if override:
        return Path(override)
    if not kaggle_root.is_dir():
        return local_root / "data/raw"
    slug = "jigsaw-agile-community-rules"
    for directory in (kaggle_root / "competitions" / slug, kaggle_root / slug):
        if all(
            (directory / name).is_file()
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        ):
            return directory
    raise FileNotFoundError(
        "Attach the official Jigsaw - Agile Community Rules Classification competition "
        "data in Kaggle's Input panel, then Save Version and Run All again."
    )


GENERATE_SUBMISSION = True
candidates = [Path.cwd(), *Path.cwd().parents]
project = next((p for p in candidates if (p / "src/jigsaw_rules").is_dir()), None)
on_kaggle = Path("/kaggle/input").is_dir()
default_output = Path("/kaggle/working") if on_kaggle else (project or Path.cwd()) / "kaggle_output"
input_root = submission_input_root(project or Path.cwd())
output_root = Path(os.environ.get("JIGSAW_KAGGLE_OUTPUT", str(default_output)))
default_cache = project / "runs/submission_cache" if project and not on_kaggle else output_root / "checkpoints"
cache_root = Path(os.environ.get("JIGSAW_SUBMISSION_CACHE", str(default_cache)))
submission_path = None
if GENERATE_SUBMISSION:
    submission_path = generate_submission(input_root, output_root, cache_root, source_sha256="2671bcad265c8034387c1e0b999b1f31e137d21bb41b9cbd8aba66a5ce1e693d")
else:
    print("Generation disabled. No CSV created, no model fitted, no upload performed.")

## Download your validated file
The link below is created only after validation and checksum verification. Click it to download your file. For files over 10 MB, use the output file browser instead of embedding a large payload. You decide whether and when to submit.

In [ ]:
from IPython.display import HTML, display
if submission_path is not None:
    display(HTML(download_link(submission_path)))
    print("Local output:", submission_path)
    print("No Kaggle submission or upload was made.")